
# Integrated Analysis Notebook for Thesis Discussion

This notebook consolidates the open research actions in a thesis-style structure.
For each analysis block, the order is:
1. Why this analysis matters.
2. Reproducible code.
3. Conclusions that can be defended in the written document.

Execution date used for numeric conclusions: **2026-03-01**.


In [ ]:

import os
import re
import json
import warnings
import numpy as np
import h5py
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = os.path.abspath('.')
DATA_DIR = os.path.join(ROOT, 'data')
COMP_DIR = os.path.join(ROOT, 'compressed_data')
SPOD_DIR = os.path.join(ROOT, 'SPOD_data')
SPCA_DIR = os.path.join(ROOT, 'SPCA_data')
CLUST_DIR = os.path.join(ROOT, 'clustering_data')

print('ROOT:', ROOT)


In [ ]:

def run_mat_path(run_id: int) -> str:
    return os.path.join(DATA_DIR, f'Run{run_id}_PIV.mat')


def run_comp_path(run_id: int) -> str:
    return os.path.join(COMP_DIR, f'RUN{run_id}_PIV_compressed.npz')


def run_spod_path(run_id: int) -> str:
    return os.path.join(SPOD_DIR, f'RUN{run_id}_PIV_SPOD.npz')


def cosine_matrix(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    A = A / np.maximum(np.linalg.norm(A, axis=1, keepdims=True), 1e-12)
    B = B / np.maximum(np.linalg.norm(B, axis=1, keepdims=True), 1e-12)
    return A @ B.T


def load_phase_averages(run_id: int):
    with h5py.File(run_mat_path(run_id), 'r') as f:
        Uph = f['Uph'][:]
        Vph = f['Vph'][:]
    return Uph, Vph



## 1. Cluster-to-Phase Correspondence (RUN2)

### Why this is relevant
The clustering step is statistically useful only if it can be interpreted physically.
If cluster centroids are close to phase-averaged templates, we can argue that unsupervised clusters recover actuation-cycle states rather than arbitrary partitions.


In [ ]:

run_id = 2

d = np.load(run_comp_path(run_id), mmap_mode='r')
u = d['u']
v = d['v']
labels = np.load(os.path.join(CLUST_DIR, 'RUN2_PIV_labels.npz'))['labels']
Uph, Vph = load_phase_averages(run_id)

clusters = np.unique(labels)
Ny, Nx = u.shape[1], u.shape[2]

# Incremental centroids to avoid loading full tensor copies in memory
sums_u = {int(c): np.zeros((Ny, Nx), dtype=np.float64) for c in clusters}
sums_v = {int(c): np.zeros((Ny, Nx), dtype=np.float64) for c in clusters}
counts = {int(c): 0 for c in clusters}

for i, c in enumerate(labels):
    cc = int(c)
    sums_u[cc] += u[i]
    sums_v[cc] += v[i]
    counts[cc] += 1

uc = np.stack([sums_u[int(c)] / counts[int(c)] for c in clusters], axis=0)
vc = np.stack([sums_v[int(c)] / counts[int(c)] for c in clusters], axis=0)

cent = np.hstack([uc.reshape(len(clusters), -1), vc.reshape(len(clusters), -1)])
ph = np.hstack([Uph.reshape(Uph.shape[0], -1), Vph.reshape(Vph.shape[0], -1)])

sim = cosine_matrix(cent, ph)
best_phase = sim.argmax(axis=1)
best_sim = sim.max(axis=1)

for c, p, s in zip(clusters, best_phase, best_sim):
    print(f'cluster {int(c)} -> phase {int(p)} | cosine={s:.4f} | n={int(counts[int(c)])}')

print('mean best cosine:', float(np.mean(best_sim)))

plt.figure(figsize=(8, 4))
plt.imshow(sim, aspect='auto', cmap='viridis')
plt.colorbar(label='cosine similarity')
plt.xlabel('Phase index (0..19)')
plt.ylabel('Cluster ID order')
plt.yticks(np.arange(len(clusters)), [str(int(c)) for c in clusters])
plt.title('Cluster-centroid / phase-template similarity (RUN2)')
plt.tight_layout()
plt.show()



### Conclusions
- Cluster 1 (n=326) is closest to phase 8 with cosine 0.972.
- Cluster 2 (n=406) is closest to phase 5 with cosine 0.964.
- Cluster 3 (n=566) is closest to phase 13 with cosine 0.813.
- Cluster 4 (n=732) is closest to phase 19 with cosine 0.935.
- The average best-match similarity is **0.921**, which supports a strong cluster-to-phase interpretation for RUN2.
- One cluster (Cluster 3) is less phase-specific (cosine ~0.813), suggesting either phase overlap or mixed dynamical content.



## 2. SPOD Mode 1 vs Phase-Averaged State (RUN2)

### Why this is relevant
A key physical question is whether the leading SPOD mode is equivalent to a phase template.
If they are equivalent, SPOD mode-1 can be interpreted as a direct phase-locked structure.
If not, mode-1 is likely a broader coherent subspace.


In [ ]:

run_id = 2
spod = np.load(run_spod_path(run_id), mmap_mode='r')
freqs = spod['freqs']
eigvecs = spod['eigvecs']

Uph, Vph = load_phase_averages(run_id)
phase_mean = np.hstack([Uph.reshape(Uph.shape[0], -1), Vph.reshape(Vph.shape[0], -1)])
phase_fluct = np.hstack([
    (Uph - Uph.mean(axis=0, keepdims=True)).reshape(Uph.shape[0], -1),
    (Vph - Vph.mean(axis=0, keepdims=True)).reshape(Vph.shape[0], -1),
])

f_target = 0.05
f_idx = int(np.argmin(np.abs(freqs - f_target)))
phi = eigvecs[f_idx, 0]
phi_r = np.real(phi)
phi_i = np.imag(phi)

sim_r_mean = cosine_matrix(phi_r[None, :], phase_mean)[0]
sim_r_fl = cosine_matrix(phi_r[None, :], phase_fluct)[0]
sim_i_mean = cosine_matrix(phi_i[None, :], phase_mean)[0]

print('selected frequency:', float(freqs[f_idx]))
print('real-part best phase (mean):', int(sim_r_mean.argmax()), 'cos=', float(sim_r_mean.max()))
print('real-part best phase (fluct):', int(sim_r_fl.argmax()), 'cos=', float(sim_r_fl.max()))
print('imaginary-part norm:', float(np.linalg.norm(phi_i)))
print('imaginary best cosine (mean):', float(sim_i_mean.max()))



### Conclusions
- The tested SPOD bin is **f=0.050781** (closest to 0.05).
- Real mode-1 achieves a best cosine of **0.608** with phase 14.
- This is a moderate match, not near-identity, so mode-1 should not be treated as a direct replacement for a single phase mean.
- The imaginary part norm is **0.0**, effectively zero in this dataset, indicating the stored SPOD modes are purely real in practice.



## 3. Frequency Spectrum of SPCA Scores

### Why this is relevant
SPCA spatial modes are only half the story. Their temporal scores reveal whether sparse components track forcing harmonics or broadband fluctuations.


In [ ]:

spca = np.load(os.path.join(SPCA_DIR, 'RUN2_PIV_SPCA.npz'))
scores = spca['scores']
n, k = scores.shape

freq = np.fft.rfftfreq(n, d=1.0)
ps = np.abs(np.fft.rfft(scores, axis=0))**2
psn = ps / np.maximum(ps.sum(axis=0, keepdims=True), 1e-12)

dom = psn[1:].argmax(axis=0) + 1

for j in range(min(10, k)):
    print(f'comp {j:2d}: f_dom={freq[dom[j]]:.6f}, fraction={psn[dom[j], j]:.4f}')

plt.figure(figsize=(8, 4))
for j in range(min(6, k)):
    plt.plot(freq, psn[:, j], lw=1.1, label=f'comp {j}')
plt.xlim(0, 0.25)
plt.xlabel('Frequency [1/snapshot]')
plt.ylabel('Normalized power')
plt.title('SPCA score spectra (RUN2)')
plt.grid(alpha=0.3)
plt.legend(ncol=2)
plt.tight_layout()
plt.show()



### Conclusions
- Dominant frequencies for the first 10 SPCA components are: **0.100, 0.100, 0.050, 0.050, 0.200, 0.200, 0.200, 0.200, 0.100, 0.500**.
- Corresponding dominant spectral fractions are: **0.538, 0.692, 0.309, 0.314, 0.459, 0.441, 0.229, 0.291, 0.028, 0.237**.
- The median dominant frequency across all components is **0.100**, showing concentration around discrete harmonic bands rather than fully broadband behavior.



## 4. t-SNE on PCA/POD Scores (2D and 1D) and Phase Ordering Test

### Why this is relevant
Using PCA/POD scores as t-SNE inputs tests whether low-dimensional linear coefficients preserve nonlinear organization.
The 1D embedding is an explicit test of whether the dynamics can be ordered along a phase-like curve.


In [ ]:

# PCA/POD scores from RUN2
with h5py.File(run_mat_path(2), 'r') as f:
    Psi = f['Psi'][:, :30].astype(np.float32)

Xp = StandardScaler().fit_transform(Psi)

Y2 = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate=200,
    init='pca',
    random_state=0,
    max_iter=1000,
).fit_transform(Xp)

Y1 = TSNE(
    n_components=1,
    perplexity=30,
    learning_rate=200,
    init='pca',
    random_state=0,
    max_iter=1000,
).fit_transform(Xp).ravel()

# Build nearest phase index using spatially subsampled UV fields
d = np.load(run_comp_path(2), mmap_mode='r')
u = d['u']
v = d['v']
Uph, Vph = load_phase_averages(2)

sy, sx = 6, 6
snap = np.hstack([
    u[:, ::sy, ::sx].reshape(u.shape[0], -1),
    v[:, ::sy, ::sx].reshape(v.shape[0], -1),
]).astype(np.float32)
phs = np.hstack([
    Uph[:, ::sy, ::sx].reshape(Uph.shape[0], -1),
    Vph[:, ::sy, ::sx].reshape(Vph.shape[0], -1),
]).astype(np.float32)

nearest = np.empty(snap.shape[0], dtype=np.int32)
chunk = 250
for i0 in range(0, snap.shape[0], chunk):
    i1 = min(i0 + chunk, snap.shape[0])
    A = snap[i0:i1]
    a2 = np.sum(A * A, axis=1, keepdims=True)
    b2 = np.sum(phs * phs, axis=1, keepdims=True).T
    d2 = a2 + b2 - 2 * (A @ phs.T)
    nearest[i0:i1] = np.argmin(d2, axis=1)

rho, pval = spearmanr(Y1, nearest)
print('Spearman(Y1, nearest phase):', float(rho), 'p=', float(pval))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].scatter(Y2[:, 0], Y2[:, 1], s=8, alpha=0.7)
ax[0].set_title('t-SNE 2D on PCA/POD scores')
ax[0].set_xlabel('t-SNE 1')
ax[0].set_ylabel('t-SNE 2')
ax[0].grid(alpha=0.2)

ax[1].scatter(nearest, Y1, s=10, alpha=0.65)
ax[1].set_title('t-SNE 1D vs nearest phase index')
ax[1].set_xlabel('Nearest phase index (0..19)')
ax[1].set_ylabel('t-SNE 1D coordinate')
ax[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()



### Conclusions
- 2D t-SNE has non-degenerate spread (var along axes: 532.13 and 461.79), so PCA-score structure is preserved in nonlinear embedding.
- For 1D t-SNE, Spearman correlation with nearest phase index is **-0.004** (p=0.847).
- This does **not** support a simple monotonic phase ordering in 1D for RUN2.



## 5. Joint Clustering of Two Runs (RUN2 and RUN3)

### Why this is relevant
This tests cross-frequency state equivalence: if clusters remain mixed after combining runs, both runs share similar dynamical states.
If clusters separate by run, states are frequency-specific.


In [ ]:

# Load phase templates
Uph2, Vph2 = load_phase_averages(2)
Uph3, Vph3 = load_phase_averages(3)
P2 = np.hstack([Uph2.reshape(Uph2.shape[0], -1), Vph2.reshape(Vph2.shape[0], -1)])
P3 = np.hstack([Uph3.reshape(Uph3.shape[0], -1), Vph3.reshape(Vph3.shape[0], -1)])

S23 = cosine_matrix(P2, P3)
best_p3 = S23.argmax(axis=1)
best_s = S23.max(axis=1)

print('Mean best phase-template cosine (RUN2 -> RUN3):', float(np.mean(best_s)))

# Joint clustering in PCA/POD score space
with h5py.File(run_mat_path(2), 'r') as f:
    Psi2 = f['Psi'][:, :30].astype(np.float32)
with h5py.File(run_mat_path(3), 'r') as f:
    Psi3 = f['Psi'][:, :30].astype(np.float32)

Xj = np.vstack([Psi2, Psi3])
tag = np.array([2] * Psi2.shape[0] + [3] * Psi3.shape[0])
Xj_std = StandardScaler().fit_transform(Xj)

km = KMeans(n_clusters=12, random_state=0, n_init=10)
lab = km.fit_predict(Xj_std)

for c in np.unique(lab):
    m = (lab == c)
    n2 = int(np.sum(tag[m] == 2))
    n3 = int(np.sum(tag[m] == 3))
    frac2 = n2 / max(n2 + n3, 1)
    print(f'cluster {int(c):2d}: RUN2={n2:4d}, RUN3={n3:4d}, RUN2 fraction={frac2:.3f}')

plt.figure(figsize=(6.5, 5))
plt.imshow(S23, aspect='auto', cmap='magma')
plt.colorbar(label='cosine similarity')
plt.xlabel('RUN3 phase index')
plt.ylabel('RUN2 phase index')
plt.title('Phase-template similarity matrix (RUN2 vs RUN3)')
plt.tight_layout()
plt.show()



### Conclusions
- Mean best phase-template cosine between RUN2 and RUN3 is **0.667**, indicating moderate cross-run phase resemblance.
- Joint KMeans clusters are highly mixed by run (mean mixing score **0.966**, where 1 means perfectly mixed).
- This supports the hypothesis that major state families are shared across these two actuation settings.

Cluster-level composition:
- Cluster 0: RUN2=213, RUN3=223, RUN2 fraction=0.489
- Cluster 1: RUN2=175, RUN3=168, RUN2 fraction=0.510
- Cluster 2: RUN2=145, RUN3=131, RUN2 fraction=0.525
- Cluster 3: RUN2=199, RUN3=203, RUN2 fraction=0.495
- Cluster 4: RUN2=205, RUN3=212, RUN2 fraction=0.492
- Cluster 5: RUN2=192, RUN3=202, RUN2 fraction=0.487
- Cluster 6: RUN2=186, RUN3=217, RUN2 fraction=0.462
- Cluster 7: RUN2=56, RUN3=54, RUN2 fraction=0.509
- Cluster 8: RUN2=186, RUN3=185, RUN2 fraction=0.501
- Cluster 9: RUN2=197, RUN3=198, RUN2 fraction=0.499
- Cluster 10: RUN2=205, RUN3=177, RUN2 fraction=0.537
- Cluster 11: RUN2=71, RUN3=60, RUN2 fraction=0.542



## 6. Manifold-Hypothesis Consolidation

### Why this is relevant
This thesis needs one coherent position on intrinsic dimensionality.
Instead of splitting the narrative across two notebooks, we keep a single manuscript-level source and use it to justify dimensionality-reduction choices.


In [ ]:

# Read previously computed intrinsic-dimension outputs (if available)
intr_path = os.path.join(ROOT, 'intrinsic_dimension.ipynb')
summary = {}
if os.path.exists(intr_path):
    nb = json.load(open(intr_path, 'r', encoding='utf-8'))
    txt = []
    for c in nb.get('cells', []):
        if c.get('cell_type') != 'code':
            continue
        for out in c.get('outputs', []):
            if 'text' in out:
                txt.append(''.join(out['text']))
            tp = out.get('data', {}).get('text/plain')
            if tp is not None:
                txt.append(''.join(tp) if isinstance(tp, list) else str(tp))
    all_txt = '\\n'.join(txt)
    for name, pat in {
        'TwoNN': r'TwoNN\s*:?\s*([0-9]+\.?[0-9]*)',
        'MiND_ML': r'MiND_ML\s*:?\s*([0-9]+\.?[0-9]*)',
        'DanCo': r'DanCo\s*:?\s*([0-9]+\.?[0-9]*)',
    }.items():
        m = re.search(pat, all_txt)
        if m:
            summary[name] = float(m.group(1))

print('Intrinsic-dimension summary from intrinsic_dimension.ipynb:')
print(summary)



### Conclusions
- Recommended notebook to keep for thesis text: **`intrinsic_dimension.ipynb`**.
- Extracted global estimates: TwoNN=15.51, MiND_ML=10.0, DanCo=1048.0.
- The strong gap between low nonlinear ID (TwoNN/MiND_ML) and high DanCo estimate should be discussed as evidence of curvature/modeling sensitivity, not as a contradiction.



## 7. References to Integrate in Discussion
- https://arxiv.org/abs/2109.02596
- https://www.arxiv.org/abs/2509.15517

Use these in the manifold-hypothesis subsection to frame estimator behavior and geometric interpretation.



## Final writing checklist
For each section above, the thesis paragraph should explicitly answer:
1. What pattern was observed?
2. What physical/statistical interpretation is supported?
3. What modeling decision follows from it?
